<a href="https://colab.research.google.com/github/bidallei/MIAAD-UACJ/blob/main/Practica_SR_FC_MovieLeans_2_AlvaroHernandez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Práctica SR Fc MovieLeans 2**

* Asignatura: Sistemas de recomendación
* Profesor: Dra. Karla Olmos Sánchez
* Alumno: **Álvaro Hernández Jarquín**
* Matrícula: 263150
* Fecha de entrega: 5 de octubre de 2026




**Objetivo de aprendizaje**

Implementar y evaluar un sistema de recomendación basado en filtrado colaborativo usuario–usuario mediante el análisis de un conjunto de datos real, el cálculo de similitudes, la predicción de preferencias y la interpretación de métricas de evaluación, considerando las implicaciones técnicas, éticas y sistémicas de las recomendaciones generadas.

**Recursos**

Google Colab y Python.

Bibliotecas Pandas, NumPy y Scikit-learn.

Conjunto de datos MovieLens 100K.

Conocimientos previos de matrices usuario–ítem, similitud coseno, vecinos y predicción de preferencias.

#Actividad 1. Cargar y explorar MovieLens

En la práctica anterior se utilizó una matriz pequeña con pocos usuarios e items. En esta actividad utilizaremos MovieLens 100K, un conjunto de datos con 100,000 calificaciones que permite trabajar con una situación más cercana a un sistema de recomendación real.

El conjunto de datos contiene información de usuarios, películas y calificaciones en una escala de 1 a 5. Está disponible en el repositorio de GroupLens.

GroupLens

**Consultar y descargar MovieLens 100K**

El archivo *u.data* contiene las calificaciones y *u.item* proporciona información sobre las películas, incluyendo sus títulos.

**Preguntas previas a la ejecución**

P1. Si el conjunto de datos contiene 100,000 calificaciones, ¿significa que contiene 100,000 usuarios? Justifica tu respuesta.
* No, significa que hay 100,000 registros de evaluación de un usuario de una película, pero pueden haber sido realizadas por un mismo usuario para varias películas.

P2. ¿Qué información necesitaríamos para construir una matriz usuario–ítem?
* Podemos usar la información de las valoraciones de cada usuario, las columnas user_id, movie_id y rating.

P3. Si dos usuarios calificaron una misma película con una puntuación de 5, ¿podemos concluir que tienen gustos similares?
* No de forma definitiva, pero sería un indicio de que coinciden en esa película. Habría que mirar más calificaciones y si tienen más títulos en común o de géneros parecidos para poder realizar una aseveración por el estilo.

P4. ¿Por qué es conveniente utilizar un DataFrame?
* Para poder trabajar de forma más rápida, ordenada y controlada, dado que con las librerías de Python en pocos comandos podemos realizar muchos procesos de forma muy rápida.

P5. ¿Qué realiza este segmento de código y porqué se deben trabajar así las columnas?
* Combina los datos del id, título, fecha de lanzamient de la película, fecha de lanzamiento del video y el url de imdb con los géneros, que están marcados con valores binarios.

columnas_peliculas = [
    "movie_id", "title", "release_date",
    "video_release_date", "imdb_url"
] + [f"genre_{i}" for i in range(19)]

**Después de ejecutar la celda**

Analiza el resultado de la ejecución.


In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

# Descargar el conjunto de datos
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

respuesta = requests.get(url, timeout=60)
respuesta.raise_for_status()

archivo_zip = zipfile.ZipFile(io.BytesIO(respuesta.content))

# Cargar las calificaciones
columnas = ["user_id", "movie_id", "rating", "timestamp"]

ratings = pd.read_csv(
    archivo_zip.open("ml-100k/u.data"),
    sep="\t",
    names=columnas
)

# Cargar los títulos de las películas
columnas_peliculas = [
    "movie_id", "title", "release_date",
    "video_release_date", "imdb_url"
] + [f"genre_{i}" for i in range(19)]

movies = pd.read_csv(
    archivo_zip.open("ml-100k/u.item"),
    sep="|",
    encoding="latin-1",
    names=columnas_peliculas
)
display(movies.head())

# Conservar únicamente el identificador y el título

movies = movies[["movie_id", "title"]]

print("Primeras calificaciones:")
display(ratings.head())

print("Primeras películas:")
display(movies.head())


,movie_id,title,release_date,video_release_date,imdb_url,genre_0,genre_1,genre_2,genre_3,genre_4,...,genre_9,genre_10,genre_11,genre_12,genre_13,genre_14,genre_15,genre_16,genre_17,genre_18
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


Primeras calificaciones:


,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Primeras películas:


,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


#Actividad 2. Analizar los datos y construir la matriz usuario–ítem

**Preguntas previas**

P6. ¿Qué instrucciones de Pandas utilizarías para determinar el número de usuarios y películas diferentes? ¿Por qué contar las filas no es suficiente?

* Podríamos usar nunique(). Contar los filas no es suficiente porque contaríamos las veces que se realizaron evaluaciones.

P7. Si tenemos 943 usuarios y 1,682 películas, ¿cuántas posiciones podría contener la matriz usuario–ítem? ¿Todas las posiciones necesariamente tienen una calificación?

* Si cada usuario calificara todas las películas, tendríamos 1,586,126 posiciones. En esta base de datos tenemos solo 100,000 calificaciones.

P8. ¿Qué diferencia existe entre una calificación de 0 y una calificación desconocida? ¿Es adecuado sustituir todos los valores desconocidos por cero?

* En MoviLeans las calificaciones posibles son de 1 a 5, así que 0 significa que no se realizó la evaluación, y la calificación desconocida es un dato que falta. Así que no sería adecuado sustituir por cero, lo mejor sería dejar esas evaluaciones como dato faltante.

P9. ¿Qué problemas puede ocasionar que algunos usuarios califiquen muchas películas y otros solamente unas pocas?
* Que tendremos datos desbalanceados. Los usuarios que realizan pocas evaluaciones pueden afectar con su calificación de forma muy significativa, pero no son confiables. Y los usuarios con muchas calificaciones pueden afectar la selección de los usuarios que se parecen.

P10. ¿Qué realizan los comandos *nunique, pivot, size, iloc, isna*?
* `nunique` cuenta los valores que no se repiten, `pivot` combina valores de dos tablas relacionados con un valor, `size` muestra el tamaño de filas y columnas de un DataFrame, `iloc` es para seleccionar las filas y columnas de un Dataframe y `isna` detecta los valores nulos y pone *True* donde hay Nan y *False* donde no hay.

P11. En términos generales ¿Qué significa dispersión?
* La proporción de películas que no fueron evaluadas por los usuarios.

**Pregunta de análisis posterior a la ejecución de la celda**

P12 ¿Cómo se interpretan los valores de *dispersión* y *densidad obtenidos*?
* La cifra 6.3% nos muestra la proporción de películas que fueron calificadas. El restante son las películas vacías, lo que quiere decir que tenemos muy pocas películas calificadas. Por eso se crean programas de descuento o recompensa a las personas que realizan evaluaciones o comparten sus datos de comportamiento, para poder crear sistemas más confiables o cercanos a la realidad.

P13 ¿De qué manera puede influir una matriz con muchas posiciones vacías en la identificación de usuarios similares?
* Va a ser más difícil tener valores entre dos usuarios, porque solo podemos usar los datos de las películas que sí calificaron. Si dos usuarios tienen pocas películas calificadas, el sistema calcula de forma poco confiable, porque son pocos datos. Dado que algunos no tienen calificadas películas parecidas tendremos pocos vecinos. También los usuarios que hayan realizado más calificaciones serán los que más influyan en el sistema de recomendación.


In [ ]:
n_usuarios = ratings["user_id"].nunique()

n_peliculas = ratings["movie_id"].nunique()

n_calificaciones = len(ratings)

print("Usuarios:", n_usuarios)
print("Películas:", n_peliculas)
print("Calificaciones:", n_calificaciones)

# Construir la matriz usuario-ítem
matriz = ratings.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

print("Dimensiones de la matriz:", matriz.shape)

display(matriz.iloc[:5, :10])

# Calcular la dispersión

total_posiciones = n_usuarios * n_peliculas

posiciones_vacias = matriz.isna().sum().sum()

dispersión = posiciones_vacias / total_posiciones

print(f"Dispersión: {dispersión:.2%}")
print(f"Densidad: {1 - dispersión:.2%}")

Usuarios: 943
Películas: 1682
Calificaciones: 100000
Dimensiones de la matriz: (943, 1682)


movie_id,1,2,3,4,5,6,7,8,9,10
user_id,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dispersión: 93.70%
Densidad: 6.30%


#Actividad 3. Separar los datos de entrenamiento y prueba

Esta actividad es importante porque introduce una diferencia respecto a las prácticas anteriores.

Cuando utilizábamos una matriz pequeña, normalmente conocíamos todas las calificaciones disponibles y calculábamos la similitud entre usuarios a partir de ellas.

Ahora necesitamos evaluar el sistema. Para ello, ocultaremos temporalmente una parte de las calificaciones y comprobaremos si el recomendador es capaz de predecirlas.

*¿Cómo evaluaremos el sistema?*

80%
Entrenamiento

Calificaciones disponibles para construir el recomendador.


20%
Prueba

Calificaciones reservadas para evaluar las predicciones.


La separación será aleatoria dentro del historial de cada usuario, conservando calificaciones para entrenar y evaluar. Por lo tanto, representa un escenario de usuarios conocidos, no una evaluación de usuarios completamente nuevos.

**Preguntas previas**

P14. ¿Por qué no debemos utilizar todas las calificaciones para construir y evaluar el sistema?
* Para tener datos contra los cuáles evaluar si el sistema funciona. Si usamos los mismos datos va a usar esos datos para dar buenos resultados y va a parecer que da buenas respuestas, pero ya las conocía.

P15. Si el sistema utiliza para construir las similitudes una calificación que posteriormente intentará predecir, ¿qué problema metodológico estaríamos cometiendo?
* Sería una fuga de datos, porque ya conocería los datos que luego intentará predecir. Para saber si un sistema da buenas calificaciones, no debe conocer las respuestas antes de predecirlas.

P16. ¿Por qué es conveniente conservar algunas calificaciones de cada usuario en los datos de entrenamiento?
* Para conocer qué usuarios son similares entre sí, así podemos comparar a los que calificaron películas para saber que coinciden.

P17. ¿Qué función cumple el parámetro `random_state`? ¿Por qué es importante que los equipos puedan reproducir sus experimentos?
* `random_state` es un número identificador para poder reproducir un experimento. Es importante poder reproducirlo para probar que los datos se obtienen en otro momento y lugar de la misma forma.

P18. ¿Qué ejecutan las instrucciones *groupby, zip, set, &*?
* `groupby` es agrupar por cierta columna, `zip` combina los elementos de un iterable en tuplas, `set` se usa para convertir en un conjunto, lo cual hace que no existan elementos repetidos y `&` une elementos que sean iguales de dos conjuntos.

**Después de ejecutar el código**

P19 ¿Cuál es el esquema (estructura) de *particiones*? Prueba con otros índices al imprimir *particiones*?
* Es una lista de tuplas y cada tupla tiene dos DataFrames.


P20 ¿Qué tipo de dato es *interaciones_train*? Escribe los primeros cinco elementos de *interacciones_train*?
* Es un conjunto (set). [(244, 509), (249, 853), (684, 781), (59, 729), (90, 42)]

P21 ¿Qué hace esta instrucción *interacciones_train & interacciones_test*?
* Nos da los elementos que están en ambos conjuntos.

P22 ¿Porqué es importante que este resultado sea 0?
* Significa que no compartieron datos del entrenamiento en la prueba, o sea, que no hubo fuga de datos.

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir las calificaciones de cada usuario
def dividir_usuario(grupo):
    entrenamiento, prueba = train_test_split(
        grupo,
        test_size=0.20,
        random_state=42
    )
    return entrenamiento, prueba

particiones = [
    dividir_usuario(grupo)
    for _, grupo in ratings.groupby("user_id")
]

print ("Particiones", particiones[0])
print ("Particiones Train", particiones[0][0])
print ("Particiones Prueba", particiones[0][1])

train = pd.concat(
    [parte[0] for parte in particiones],
    ignore_index=True
)

test = pd.concat(
    [parte[1] for parte in particiones],
    ignore_index=True
)

print("Número de datos de entrenamiento: ", len(train))
print("Datos de entrenamiento", train)
print("Número de datos de prueba: ", len(test))
print("Datos de prueba", test)

# Verificar que una interacción no aparezca
# simultáneamente en entrenamiento y prueba


interacciones_train = set(
    zip(train["user_id"], train["movie_id"])
)


interacciones_test = set(
    zip(test["user_id"], test["movie_id"])
)

print(
    "Interacciones compartidas:",
    len(interacciones_train & interacciones_test)
)

Particiones (       user_id  movie_id  rating  timestamp
3909         1       177       5  876892701
37810        1       158       3  878542699
21152        1       179       3  875072370
43263        1        41       2  876892818
4177         1       174       5  875073198
...        ...       ...     ...        ...
38751        1       257       4  874965954
8745         1       259       1  875692979
14025        1       241       4  878543133
96699        1       152       5  878542589
13165        1        91       5  876892636

[217 rows x 4 columns],        user_id  movie_id  rating  timestamp
3733         1        31       3  875072144
15932        1        39       4  875072173
9382         1       163       4  875072442
17863        1       226       3  878543176
42456        1       169       5  878543541
20153        1        38       3  878543075
48078        1       238       4  875072235
4963         1       225       2  878542738
25721        1       141       3  8785

In [ ]:
type(particiones[0])

tuple

In [ ]:
type(particiones[0][0] )

pandas.core.frame.DataFrame

In [ ]:
type(interacciones_train)

set

In [ ]:
list(interacciones_train)[:5]

[(244, 509), (249, 853), (684, 781), (59, 729), (90, 42)]

#Actividad 4. Construir la matriz de entrenamiento y preparar los datos para Machine Learning

En esta etapa se construirá una nueva matriz usuario–ítem, pero utilizando exclusivamente las calificaciones del conjunto de entrenamiento.

También se calculará la media de las calificaciones de cada usuario.
Recuerda que dos usuarios pueden utilizar la escala de calificación de manera diferente: uno puede calificar casi todas las películas con 4 o 5, mientras que otro puede utilizar frecuentemente calificaciones de 2 y 3.

Para calcular la similitud se utilizará las desviaciones respecto a la media de cada usuario, en lugar de comparar directamente las calificaciones originales.

**Investiga antes de la ejecución:**

P23. ¿Qué diferencia existe entre calcular la similitud utilizando las calificaciones originales y utilizar las desviaciones respecto a la media de cada usuario?
* Cuando usamos las calificaciones originales hace que la valoración de un usuario sea demasiado relevante. Si usamos las desviaciones comparadas con la media de ese usuario, bajamos la diferencia lo que disminuye el impacto cuando se califican de forma extrema, muy bajas o muy altas.


P24. Si un usuario tiene una calificación media de 4 y asigna una calificación de 5 a una película, ¿cuál es su desviación respecto a la media?
* Su desviación es de 1 positivo.

P25. ¿Por qué es importante que las medias de los usuarios se calculen únicamente con los datos de entrenamiento?
* Si usamos datos que no sean de entrenamiento habría fuga de datos.


**Después de la ejecución:**

P26. ¿Qué tipo de dato es *medias_usuario*? Escribe los primeros 5 datos
`medias_usuario` es una Serie de pandas,

| user_id | rating |
|:-------:|:--------------:|
| 1       | 3.635945       |
| 2       | 3.734694       |
| 3       | 2.720930       |
| 4       | 4.315789       |
| 5       | 2.942857       |

P27. ¿Qué tipo de dato es *media_global*? Escribe su valor
`media_global` es 'numpy.float64'
3.529986560996747

P28. En la matriz normalizada, ¿qué significa un valor de 0 correspondiente a una calificación conocida? ¿Qué diferencia existe respecto a un valor faltante que representamos computacionalmente mediante cero?

Un 0 significa que la calificación de una película es la misma que la media de ese usuario.
La diferencia es que cuando hay un valor faltante, el usuario no realizó la evaluación de esa película, pero el cero en la desviación es una coincidencia exacta con su promedio.


In [ ]:
# Calcular la media de cada usuario
medias_usuario = train.groupby("user_id")["rating"].mean()

# Calcular la media global como valor de respaldo
media_global = train["rating"].mean()

# Completen el código con los comandos sub, pivot y fillna
# Construir la matriz de entrenamiento

matriz_train = train.pivot(
    index="user_id",
    columns="movie_id",
    values="rating"
)

# Restar la media correspondiente a cada usuario

matriz_centrada = matriz_train.sub(
    medias_usuario,
    axis=0
)

# Sustituir las posiciones faltantes por cero
# exclusivamente para el cálculo de similitud

matriz_ml = matriz_centrada.fillna(0)

print("Matriz original:")
display(matriz_train.iloc[:5, :8])

print("Matriz centrada:")
display(matriz_centrada.iloc[:5, :8])

print("Matriz para el algoritmo:")
display(matriz_ml.iloc[:5, :8])

print("Media global:", round(media_global, 3))


Matriz original:


movie_id,1,2,3,4,5,6,7,8
user_id,,,,,,,,
1,5.0,NaN,4.0,NaN,3.0,5.0,4.0,1.0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN


Matriz centrada:


movie_id,1,2,3,4,5,6,7,8
user_id,,,,,,,,
1,1.364055,NaN,0.364055,NaN,-0.635945,1.364055,0.364055,-2.635945
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1.057143,0.057143,NaN,NaN,NaN,NaN,NaN,NaN


Matriz para el algoritmo:


movie_id,1,2,3,4,5,6,7,8
user_id,,,,,,,,
1,1.364055,0.000000,0.364055,0.0,-0.635945,1.364055,0.364055,-2.635945
2,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
5,1.057143,0.057143,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


Media global: 3.53


In [ ]:
type(medias_usuario)

pandas.core.series.Series

In [ ]:
medias_usuario.head()

,rating
user_id,
1,3.635945
2,3.734694
3,2.720930
4,4.315789
5,2.942857


In [ ]:
print(type(medias_usuario))
print(medias_usuario.head())

<class 'pandas.core.series.Series'>
user_id
1    3.635945
2    3.734694
3    2.720930
4    4.315789
5    2.942857
Name: rating, dtype: float64


In [ ]:
print(type(media_global))
print(media_global)

<class 'numpy.float64'>
3.529986560996747


#Actividad 5. Identificar usuarios similares utilizando Machine Learning

En las prácticas anteriores calcularon la similitud coseno manualmente y seleccionaron a los usuarios con mayor similitud.

En esta práctica se utilizará *NearestNeighbors*, de *Scikit-learn*, para automatizar la búsqueda de vecinos.

El algoritmo utilizará la distancia coseno. Para convertirla en una medida de similitud:

$s(u,v)=1-d_{\cos}(u,v)$

Con los vectores que se utilizan, el resultado puede ser positivo, cero o negativo.

Una similitud positiva representa patrones de preferencias alineados según esta medida.

Para simplificar la práctica, se seleccionarán hasta 30 vecinos con similitud positiva para cada usuario.

En estas dos últimas celdas el código está completo para que se concentren en analizar el código

**Investiga antes de ejecutar el código**

P29. ¿Qué diferencia existe entre una medida de distancia y una medida de similitud? Si dos usuarios son muy similares, ¿esperarías una distancia coseno pequeña o grande?
* La distancia mide la separación entre dos elementos, entonces, si está cerca son parecidos y si está lejos son más diferentes. Y la similitud mide con un valor cuánto se parecen, si es pequeño el valor, es poco su parecido y si es grande, su parecido es mayor.

P30. ¿Qué ventajas ofrece utilizar una herramienta de Machine Learning frente a calcular manualmente la similitud entre todos los usuarios?
* Se pueden evitar errores en el cálculo, bugs, se puede hacer más escalable, las instrucciones ya están hechas, solo se deben completar parámetros.


P31. ¿Por qué debemos evitar que un usuario sea seleccionado como su propio vecino?
* Porque al tener los mismos datos solo aumentaría su sesgo en sus mismas calificaciones. Tendría fuga de información, porque se usaría en el entrenamiento y en la prueba. Se perdería la oportunidad de relacionar con otros vecinos, menos exactos, pero más reales.

P32. ¿Qué consecuencias podría tener seleccionar solamente 2 vecinos frente a seleccionar 30? ¿Por qué más vecinos garantizan mejores recomendaciones?
* Cuando tenemos pocos datos, en general, pueden afectar mucho la recomendación, porque los valores se promediarán. Si tenemos más vecinos podemos tener un valor más acertado. Sin embargo, no es una garantía, a menos que hayamos realizado una selección adecuada con vecinos que tengan valores parecidos.

P33. ¿Por qué una similitud elevada, calculada a partir de muy pocas películas compartidas, podría conducir a una predicción poco fiable?
* Porque si usamos la similitud de pocas películas puede ser que no indique que son vecinos confiables, entre más películas sean las que calificaron y que tienen valores en común, esa probabilidad es más grande y la predicción sería más fiable.


P34. ¿Qué es *csr_matrix*?
* Es un tipo de matriz de Scipy que no guarda los valores nulos, con lo que ahorra mucho espacio.

**Después de ejecutar la celda:**

P35. Escribe los primeros 5 elementos de X
* [ 1.3640553  0.3640553 -0.6359447  1.3640553  0.3640553]

P36. ¿Por qué se debe sumar 1 a K en *n_neighbors = K + 1* ?
* Porque si no el vecino más cercano sería él mismo, así que se produciría fuga de datos.

P37. ¿Qué tipo de datos son *distancias* e *indices*? Escribe los primeros 5 elementos de cada uno.
* distancias e indices son <class 'numpy.ndarray'>
distancias: [0.         0.8060955  0.83108824 0.83138282 0.83651226]
indices: [  0 690  58 362 565]


P38. ¿Qué recorre el ciclo for externo?
* Cada usuario de la matriz


P39. ¿Qué recorre el ciclo for interno?
* Los 30 vecinos del usuario


P40. ¿Para qué se realiza esta instrucción *vecinos[int(user_id)] = lista_vecinos[:K]* ?
* Para obtener en el diccionario *vecinos* la lista de los 30 vecinos del usuario y se guardan relacionadas con la clave *user_id*


P41. ¿Qué tipo de dato es *vecinos*?
* Es un diccionario.


Pregunta de Easter Egg: ¿Cuál es tu personaje de ánime o caricatura favorito?
* En la actividad anterior escogí a Pichu, ahora cambiaré por Apa de la leyenda de Aang, un especie de Bisonte volador.


In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

# Convertir la matriz a una representación dispersa
X = csr_matrix(matriz_ml.to_numpy())

# Crear el modelo de vecinos cercanos
modelo = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

# Ajustar el modelo
modelo.fit(X)

# Identificar los 30 vecinos más cercanos
K = 30

distancias, indices = modelo.kneighbors(
    X,
    n_neighbors=K + 1
)

usuarios = matriz_ml.index.to_numpy()

# Guardar los vecinos de cada usuario
vecinos = {}

for posicion, user_id in enumerate(usuarios):

    lista_vecinos = []

    for distancia, indice in zip(
        distancias[posicion],
        indices[posicion]
    ):

        vecino_id = int(usuarios[indice])
        similitud = 1 - distancia

        # Excluir al propio usuario y similitudes no positivas
        if vecino_id != user_id and similitud > 0:

            lista_vecinos.append(
                (vecino_id, float(similitud))
            )

    vecinos[int(user_id)] = lista_vecinos[:K]

# Mostrar los cinco vecinos principales del usuario 1
usuario_ejemplo = 1

tabla_vecinos = pd.DataFrame(
    vecinos[usuario_ejemplo][:5],
    columns=["vecino_id", "similitud"]
)

display(tabla_vecinos)


,vecino_id,similitud
0,691,0.193905
1,59,0.168912
2,363,0.168617
3,566,0.163488
4,880,0.163477


In [ ]:
print(X.data[:5])

[ 1.3640553  0.3640553 -0.6359447  1.3640553  0.3640553]


In [ ]:
print(type(distancias))
print(type(indices))
print(distancias[0][:5])
print(indices[0][:5])

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
[0.         0.8060955  0.83108824 0.83138282 0.83651226]
[  0 690  58 362 565]


In [ ]:
print(type(vecinos))

<class 'dict'>


# Actividad 6. Predecir una calificación utilizando los vecinos


Ya conocemos qué usuarios tienen patrones de preferencias similares. Sin embargo, todavía no se ha generado ninguna recomendación.

Para hacerlo, se requiere estimar cuánto podría gustarle una película a un usuario.

Se utilizará una predicción basada en la media del usuario y las desviaciones de sus vecinos:


$$
\hat r_{u,i} =
\bar r_u +
\frac{
\sum_{v \in N_i(u)} s(u,v)\left(r_{v,i}-\bar r_v\right)
}{
\sum_{v \in N_i(u)} |s(u,v)|
}
$$

Donde:

- $\hat r_{u,i}$: calificación predicha para el usuario $u$ y la película $i$.
- $\bar r_u$: media de las calificaciones del usuario objetivo.
- $N_i(u)$: vecinos seleccionados que efectivamente calificaron la película.
- $s(u,v)$: similitud entre el usuario objetivo y cada vecino.
- $r_{v,i}$: calificación que el vecino asignó a la película.
- $\bar r_v$: media de las calificaciones del vecino.

En esta práctica se utilizarán solamente similitudes positivas.

Si ninguno de los vecinos seleccionados calificó la película, se utilizará la media del usuario como predicción de respaldo.

**Investiga antes de ejecutar el código**

P43. ¿Qué hacen los comandos *get, at, loc, notna y abs*?
* `get `busca una clave en un diccionario y si no está ese valor no arroja un error. `at` nos ayuda a revisar un valor de un DataFrame de acuerdo a la etiqueta que elegimos. `loc` nos permite ir a los valores de una celda o fila o columna. `notna` regresa True si un valor no es nulo y False si sí lo es. `abs` nos da el valor absoluto de una número.

**Análisis para después de ejecutar el código**

P44. ¿Qué tipo de dato es *rating_vecino*? Escribe sus primeros 5 elementos
* Si el usuario calificó la película *rating_vecino* es 'numpy.float64'.
Usando al usuario 3 y la película 50:
688 → nan float64
142 → nan float64
783 → nan float64
369 → 5.0 float64
46 → 4.0 float64

P45. ¿Qué tipo de dato es *media vecino*?
* La *media vecino* es 'numpy.float64'.

P46. ¿Qué hace la instrucción

*numerador += similitud  (
                rating_vecino - media_vecino
            )*

* Nos da la suma ponderada de las desviaciones de todos los vecinos que sí calificaron la película

P47. Si el usuario objetivo tiene 30 vecinos, ¿debemos utilizar las calificaciones de los 30 para predecir su preferencia por una película? ¿Qué sucede si solamente tres vecinos la calificaron?
* Solo se utilizan los vecinos que sí calificaron la película. Si solo fueran tres vecinos, con esos se realiza el cálculo.

P48. ¿Por qué utilizamos la desviación de la calificación del vecino respecto a su propia media?
* Porque cada usuario tiene su propia escala, así se pondera su calificación respecto a su propio promedio, y se quitan valores muy disparados, en caso de que existan.

P49. ¿Qué significa que el sistema no encuentre vecinos que hayan calificado una película? ¿Cómo podríamos generar una predicción en ese caso?
Significa que ningún vecino del conjunto de entranmiento calificó esa película. Podemos usar la media del usuario. En caso de que no tuviera ninguna valoración el usuario, podríamos usar la media global,.

P50. Si el resultado de la predicción es 5.4, ¿es válido conservarlo cuando la escala de calificaciones va de 1 a 5?
* No es válido, porque nuestras calificaciones van de 1 a 5, así que se debe corregir, valores mayores de 5 bajan a 5 y valores menores de 1 suben a 1.

P51. ¿Qué diferencia existe entre una calificación observada y una calificación predicha? ¿Qué riesgos existen si el sistema presenta ambas como si fueran equivalentes?
* La observada es una que realizó un usuario real y su valor solo puede estar entre 1 y 5. En las predicciones el modelo hace la estimación y no sabemos si sea buena o no. El valor es un número en decimales.

Pregunta de Easter Egg: Escribe el nombre de tu película favorita e indica por qué te gusta

* Cambiaré mi respuesta de esta actividad también, ahora será Vatel, una película de la corte francesa en la que el "artista" que dirige todos los eventos de una visita del Rey de Francia demuestra su gran ingenio y contacto con las personas que estaban bajo su mando. La actuación de Gerard Depardieu es genial.

In [ ]:
def predecir_rating(user_id, movie_id):

    # Calificación media del usuario objetivo
    media_usuario = medias_usuario.get(
        user_id,
        media_global
    )

    # Si el usuario no tiene vecinos o la película
    # no está en entrenamiento, utilizar su media
    if user_id not in vecinos or movie_id not in matriz_train.columns:
        return float(media_usuario), 0

    numerador = 0.0
    denominador = 0.0
    n_apoyos = 0

    # Revisar los vecinos seleccionados
    for vecino_id, similitud in vecinos[user_id]:

        rating_vecino = matriz_train.at[
            vecino_id, movie_id
        ]

        # Utilizar solamente calificaciones conocidas
        if pd.notna(rating_vecino):

            media_vecino = medias_usuario.loc[vecino_id]

            numerador += similitud * (
                rating_vecino - media_vecino
            )

            denominador += abs(similitud)
            n_apoyos += 1

    # Si ningún vecino calificó la película
    if denominador == 0:
        return float(media_usuario), 0

    # Predicción colaborativa
    prediccion = media_usuario + (
        numerador / denominador
    )

    # Respetar la escala de calificación
    prediccion = np.clip(prediccion, 1, 5)

    return float(prediccion), n_apoyos


# Ejemplo de predicción
usuario_ejemplo = 1
pelicula_ejemplo = 50

prediccion, apoyos = predecir_rating(
    usuario_ejemplo,
    pelicula_ejemplo
)

print("Usuario:", usuario_ejemplo)
print("Película:", pelicula_ejemplo)
print("Calificación predicha:", round(prediccion, 3))
print("Vecinos que aportaron:", apoyos)


Usuario: 1
Película: 50
Calificación predicha: 4.977
Vecinos que aportaron: 21


In [ ]:
usuario_ejemplo = 3
pelicula_ejemplo = 50

for vecino_id, sim in vecinos[usuario_ejemplo][:5]:
    r = matriz_train.at[vecino_id, pelicula_ejemplo]
    print(vecino_id, "→", r, type(r).__name__)

688 → nan float64
142 → nan float64
783 → nan float64
369 → 5.0 float64
46 → 4.0 float64


# Actividad 7. Generar recomendaciones personalizadas

Hasta ahora se ha trabajo en el código para predecir cuánto podría gustarle una película a un usuario.

Sin embargo, la predicción de una calificación no es lo mismo que la generación de una recomendación.

Para recomendar películas debemos identificar:
- cuáles no han sido vistas o calificadas por el usuario,
- estimar sus preferencias y
- seleccionar aquellas con las puntuaciones predichas más altas.

En esta actividad se generarán cinco recomendaciones para el usuario 1.

Para evitar seleccionar películas basándose únicamente en la media del usuario, se exigirá que al menos dos vecinos hayan aportado una calificación a cada recomendación.

**Preguntas previas**

P52. Si una película tiene una calificación predicha de 4.9, pero el usuario ya la calificó anteriormente, ¿debería aparecer en su lista de recomendaciones?
* No debería haerlo. La idea de hacer una recomendación es que el usuario conozca cosas que no ha visto o calificado. Si ya la calificó, el sistema ya sabe qué opina.

P52. ¿Qué instrucciones de Python o Pandas utilizarías para ordenar las películas de mayor a menor calificación predicha?
* Usaría `sort_values(by="prediccion", ascending=False)`.

P53. ¿Por qué resulta útil conocer cuántos vecinos contribuyeron a la predicción de cada película?
* Porque así sabría cuánta evidencia respalda la predicción. Una predicción con 1 vecino es muy inestable y puede reflejar el gusto de una sola persona. Una predicción con 15 vecinos nos brinda un resultado más estable. Por eso es útil conocer cuántos vecinos contribuyeron a la predicción.

P54. Si dos películas obtienen una predicción de 4.8, pero la primera se estimó con un solo vecino y la segunda con 15, ¿qué diferencia existe entre ambas estimaciones? ¿Podemos garantizar que la segunda predicción sea correcta?
* La primera con 1 vecino es muy frágil ya que podría estar sesgada por el gusto de esa persona. La segunda con 15 vecinos es mucho más robusta y estable. De todas formas no podemos garantizar que la segunda sea correcta, nada más que es más confiable estadísticamente. Por ejemplo, si los 15 vecinos comparten un sesgo, podría ser incorrecta

P55. ¿Qué efecto podría tener sobre las recomendaciones exigir que muchas personas hayan calificado una película?
* Exigir muchos calificaciones va a reducir el catálogo de recomendaciones y provocará que solo aparezcan películas muy populares o muy vistas. El sistema estaría sesgado hacia lo más visto y deja fuera a películas nuevas o con pocas calificaciones, que quizás podrían gustarle al usuario.

P56. ¿Qué realiza los comandos *by*, *merge*,*loc* y *set* ?
* `by` es el parámetro de `sort_values` para indicar cuáles columnas tomar como base al ordenar.
* `merge` une dos *DataFrames* a partir de una columna en común.
* `loc` nos da acceso a las filas o columnas con base en una etiqueta.
* `set` crea un conjunto sin duplicados.


In [ ]:
def recomendar_peliculas(user_id, n=5, min_apoyos=2):

    # Identificar las películas calificadas por el usuario
    vistas = set(
        train.loc[
            train["user_id"] == user_id,
            "movie_id"
           ]
   )

    print("vistas", list(vistas)[0:5])

    recomendaciones = []

    # Revisar las películas disponibles
    for movie_id in matriz_train.columns:

        # No recomendar películas ya conocidas
        if movie_id in vistas:
            continue

        # Estimar la preferencia del usuario
        prediccion, apoyos = predecir_rating(
            user_id,
            movie_id
        )

        # Conservar películas con suficiente apoyo
        if apoyos >= min_apoyos:
            recomendaciones.append({
                "movie_id": movie_id,
                "prediccion": prediccion,
                "apoyos": apoyos
            })

    #   print("recomendaciones", recomendaciones)

    # Convertir los resultados en un DataFrame
    resultado = pd.DataFrame(recomendaciones)

    print("=======")
    print("resultados")
    print (resultado.head())

    if resultado.empty:
        return resultado

    # Ordenar por calificación predicha
    resultado = resultado.sort_values(
         by =["prediccion", "apoyos"],
         ascending=[False, False]
     )


    # Seleccionar las mejores n recomendaciones
    resultado = resultado.head(n)

    # Incorporar los títulos de las películas

    resultado = resultado.merge(
       movies,
       on="movie_id",
       how="left"
    )

    return resultado[
        ["movie_id", "title", "prediccion", "apoyos"]
    ]


# Generar cinco recomendaciones
recomendaciones_1 = recomendar_peliculas(
    user_id=1,
    n=5
)

display(recomendaciones_1)


vistas [1, 3, 5, 6, 7]
resultados
   movie_id  prediccion  apoyos
0         2    3.577109      15
1         4    3.976362      16
2        17    3.451272      12
3        23    4.768106      12
4        24    3.777649      17


,movie_id,title,prediccion,apoyos
0,408,"Close Shave, A (1995)",5.0,6
1,340,Boogie Nights (1997),5.0,4
2,853,Braindead (1992),5.0,3
3,1019,"Die xue shuang xiong (Killer, The) (1989)",5.0,3
4,646,Once Upon a Time in the West (1969),5.0,2


# Actividad 8. Comparar recomendaciones para diferentes usuarios

Un sistema de recomendación debe adaptar sus resultados a diferentes perfiles de usuario.

En esta actividad se compararán las recomendaciones generadas para tres usuarios y se analizará si el sistema está produciendo listas diferentes o si recomienda sistemáticamente las mismas películas.

Preguntas previas
P57. Si dos usuarios tienen vecinos diferentes, ¿necesariamente recibirán recomendaciones distintas?
* No necesariamente. Dos usuarios con vecinos diferentes podrían recibir las mismas películas si se da el caso que esas películas son vistas y varios vecinos las calificaron alto. Generalmente, si los vecinos y sus perfiles son diferentes, las listas también lo serán.

P58. ¿Qué podría indicar que todos los usuarios reciban prácticamente las mismas cinco películas?
* Que el sistema no es personalizado, que nada más está tomando en cuenta la popularidad. Esto puede pasar si los vecinos de parecen mucho o si hay pocos vecinos.

P59. ¿Por qué un sistema que recomienda únicamente películas muy populares podría resultar útil para algunos usuarios, pero limitado para otros?
* Este tipo de recomendación está bien cuando no tenemos datos del usuario, para un inicio en frío. La parte que limita es conocer recomendaciones fuera lo usual, así se recomienda y se refuerza dentro del mismo círculo.

P60. ¿De qué manera podríamos comprobar si las recomendaciones son verdaderamente personalizadas?
* Podríamos hacer una comparación de las listas entre usuarios para ver que porcentaje comparten. Si todo es alto el porcentaje se están copiando. La otra forma sería comparar si las recomendaciones se parecen a las más populares.


In [ ]:
usuarios_ejemplo = [1, 10, 100]

resultados_usuarios = {}


# Completa el código para llamar a la función recomendar_peliculas
for user_id in usuarios_ejemplo:

    recomendaciones = recomendar_peliculas (
        user_id = user_id ,
        n = 5
    )

    resultados_usuarios[user_id] = recomendaciones

    print(f"\nRecomendaciones para el usuario {user_id}")

    display(recomendaciones)


# Calcular cuántas películas comparten dos usuarios
peliculas_u1 = set(
    resultados_usuarios[1]["movie_id"]
)

peliculas_u10 = set(
    resultados_usuarios[10]["movie_id"]
)

coincidencias = peliculas_u1.intersection(peliculas_u10)

print(
    "Películas compartidas entre los usuarios 1 y 10:",
    len(coincidencias)
)


vistas [1, 3, 5, 6, 7]
resultados
   movie_id  prediccion  apoyos
0         2    3.577109      15
1         4    3.976362      16
2        17    3.451272      12
3        23    4.768106      12
4        24    3.777649      17

Recomendaciones para el usuario 1


,movie_id,title,prediccion,apoyos
0,408,"Close Shave, A (1995)",5.0,6
1,340,Boogie Nights (1997),5.0,4
2,853,Braindead (1992),5.0,3
3,1019,"Die xue shuang xiong (Killer, The) (1989)",5.0,3
4,646,Once Upon a Time in the West (1969),5.0,2


vistas [1, 513, 4, 518, 519]
resultados
   movie_id  prediccion  apoyos
0         2    3.518671       2
1         3    2.599001       2
2         5    3.798680       4
3         8    4.595714      12
4        10    4.208930       2

Recomendaciones para el usuario 10


,movie_id,title,prediccion,apoyos
0,180,Apocalypse Now (1979),5.0,11
1,515,"Boot, Das (1981)",5.0,7
2,647,Ran (1985),5.0,7
3,293,Donnie Brasco (1997),5.0,4
4,493,"Thin Man, The (1934)",5.0,4


vistas [258, 905, 266, 908, 269]
resultados
   movie_id  prediccion  apoyos
0       100    3.939593       2
1       237    3.140533       2
2       242    3.900284       3
3       243    2.687814       5
4       245    2.910110      12

Recomendaciones para el usuario 100


,movie_id,title,prediccion,apoyos
0,1176,Welcome To Sarajevo (1997),4.442901,2
1,361,Incognito (1997),4.201385,2
2,984,Shadow Conspiracy (1997),4.198687,2
3,750,Amistad (1997),3.954945,11
4,1025,Fire Down Below (1997),3.945755,2


Películas compartidas entre los usuarios 1 y 10: 0


#Preguntas de análisis

P61. ¿Qué diferencias observas entre las recomendaciones de los tres usuarios? Fundamenta tu respuesta utilizando los resultados obtenidos.
* Ninguna de las películas de las 3 recomendaciones es la misma, así que todas las listas son personalizadas. Las películas de los primeros dos usuarios incluyen películas más antiguas y el último tiene puras de 1997.

P62. ¿Por qué dos usuarios pueden recibir la misma película como recomendación, aunque sus perfiles de preferencias sean diferentes?
* Porque puede tratarse de p0elículas muy populares o que tienen vecinos similares que calificaron altas esas películas.

P63. ¿Qué información necesitarías para determinar si las recomendaciones responden a preferencias individuales o principalmente a la popularidad de las películas?
* Tener claro cuáles son las más populares y siempre estar comparando las recomendaciones con esta lista; los géneros preferidos de cada usuario según sus calificaciones más altas; y, la comparación de las calificaciones del usuario con las recomendaciones, para ver si coinciden.